In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
v1.shape

(384,)

In [4]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
v1.dot(dv)

np.float32(0.3233239)

In [6]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [7]:
v1.dot(v2)

np.float32(-0.14271769)

# Embed all the documents

In [8]:
import sys
import os

# Add the parent directory (src) to the system path
# The '..' tells it to look one folder up from where the notebook is currently running
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [9]:
from ingest import load_faq_data
documents = load_faq_data()

In [10]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [11]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vector = model.encode(batch)
    vectors.extend(batch_vector)

  0%|          | 0/27 [00:00<?, ?it/s]

In [12]:
len(vectors)

1350

In [13]:
import numpy as np
X = np.array(vectors) #Faster than uisng for loop(matrix vector multiplication)

In [14]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [15]:
scores = X.dot(v_query)

In [16]:
# With for 
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [17]:
#Best match
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629409))

In [18]:
#Dcouments return
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [19]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [20]:
# Read the top 5 documents and their scores
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629409
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579371
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related

In [21]:
top5 = np.argsort(-scores)[:5]

TypeError: bad operand type for unary -: 'list'

## Combining Vector search with minisearch

In [22]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [23]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [24]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [ ]:
#Filtering with minsearch
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

### Vector Search

In [25]:
from rag_helper import RAGBase
from ingest import build_index
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

index = build_index(documents)


assistant = RAGBase(
    index=index,
    llm_client=client,
)

In [26]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still sign up for the program. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [31]:
class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            filter_dict=filter_dict,
            num_results=num_results,
        )

In [34]:
assistant_vector = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client,
)

In [35]:
assistant_vector.rag("the program has already begun, can I still sign up?")

'Yes, you can still sign up for the course, but if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

### Add Sqlitesearch

In [36]:

from sqlitesearch import VectorSearchIndex


vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [38]:
vs_index.close()

In [37]:
vs_index.fit(vectors, documents)

ValueError: Index already contains documents. Use clear() to reset the index or add() to append documents.